# Deploy and inference Jina Embeddings v5 Omni Small with Google Cloud Marketplace Virtual machine

This notebook demonstrates how to deploy a Jina Embeddings v5 Omni Small model-powered [Google Cloud Marketplace Virtual machine](https://console.cloud.google.com/marketplace/product/jinaai-public/jina-embeddings-v5-omni-small) and perform inference with the application within the virtual machine.

Jina Embeddings v5 Omni Small embeds text, images, video, audio and PDF into one shared vector space. It supports a context length of 32,768 tokens, produces 1024-dimensional embeddings that can be truncated down to 32 dimensions with Matryoshka representation learning, and offers four task adapters: `retrieval`, `text-matching`, `clustering` and `classification`.

## Deploy the virtual machine

To deploy the virtual machine, start by consulting the [official deployment guide](https://cloud.google.com/marketplace/docs/deploy-through-CLI#use-snippet). This document provides comprehensive steps for the deployment process.

It's worth mentioning that in the Basics tab of the VM setup, you will need to provide several details about the VM. 

The recommended machine type is `g2-standard-8` (8 vCPU, 32 GB memory), which comes with 1 [NVIDIA L4](https://cloud.google.com/compute/docs/gpus?_gl=1*l4wato*_ga*MTI3NjQzMzcwNy4xNzI3NTg1MDYx*_ga_WH2QY8WWF5*MTczMDA4MDMzMC42NC4xLjE3MzAwODQ4NTQuMzcuMC4w#general_comparison_chart) GPU with 24 GB of memory. The L4 is intrinsic to the G2 machine family, so you do not need to attach it separately. The boot disk size should be minimum 50 GB. For some machine sizes you might need to adjust the allowed quota to ensure access.

L4 capacity is scarce and the deployment form defaults to the `us-central1-a` zone. If deployment fails with a stockout error such as `ZONE_RESOURCE_POOL_EXHAUSTED`, retry in a different zone that offers G2 machines.

For the other tabs, you can leave most settings as default or adjust them to fit your needs.

<img src="images/deploy_embeddings_v5_omni_small_vm.png" width="50%" height="50%" style="max-width: 600px">

Once the deployment of the VM is complete, proceed to the resource group created for your deployment to verify the resources that have been established. 

Within the resource group, you'll find the public IP through which you can access your application within the VM. The application serves plain HTTP on port 80.

Please note that the application within the VM will be unavailable immediately after deployment due to necessary model loading process **We recommend waiting at least 7 minutes before using the application.** You can poll `http://<your public IP address>/health` to check whether the model has finished loading.

# Perform inference with the application

The Python example below demonstrates how to perform real-time inference using the public IP address of the deployed virtual machine.

Embeddings are served on `POST /v1/embeddings`. A `POST /predict` endpoint is also available for Vertex AI conventions.

In [ ]:
import json

import requests

url = "http://<Insert here your public IP address>/v1/embeddings"  # For example: http://20.84.48.180/v1/embeddings
headers = {"Content-Type": "application/json"}


def embed_text():
    json_data = {
        "input": "A beautiful sunset over the beach",
        # Select the downstream task for which the embeddings will be used. The model
        # will return the optimized embeddings for that task. Should be one of
        # `retrieval.query`, `retrieval.passage`, `text-matching`, `clustering` or `classification`.
        "task": "retrieval.query",
        # Output embedding size. The model is trained at 1024 dimensions and supports
        # Matryoshka truncation to 512, 256, 128, 64 and 32.
        "dimensions": 1024,
    }

    response = requests.post(url, headers=headers, data=json.dumps(json_data))
    print(response.json())


def embed_multimodal():
    # The model maps every modality into the same vector space, so text and image
    # embeddings from the same request are directly comparable.
    json_data = {
        "input": [
            {"text": "A beautiful sunset over the beach"},
            {"image": "https://i.ibb.co/nQNGqL0/beach1.jpg"},
        ],
        "task": "retrieval.passage",
        "dimensions": 1024,
    }

    response = requests.post(url, headers=headers, data=json.dumps(json_data))
    print(response.json())


embed_text()
embed_multimodal()